In [74]:
# ============================================================
# SETUP, IMPORTS, CONFIGURATION, AND HELPER FUNCTIONS
# ============================================================

import pandas as pd
import numpy as np
import re
import json
import time

from pathlib import Path
from difflib import SequenceMatcher
from tqdm import tqdm

# ------------------------------------------------------------
# FILE CONFIGURATION
# ------------------------------------------------------------

FILE_PATH = Path("Inventory Incident Report FINAL.xlsx")

# Fallback if Colab stores the file name with URL encoding
if not FILE_PATH.exists():
    FILE_PATH = Path("Inventory%20Incident%20Report%20FINAL.xlsx")

SHEET_NAME = "Sheet4"

VALID_INCIDENT_TYPES = ["RMA", "Work Order"]

BUSINESS_COST_THRESHOLD = 10000


# ------------------------------------------------------------
# GENERAL CLEANING HELPERS
# ------------------------------------------------------------

def clean_column_names(df):
    """
    Converts column names to snake_case.
    Example: 'Incident Date' becomes 'incident_date'.
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    return df


def clean_text(value):
    """
    Standardises text fields by stripping whitespace
    and converting blank strings to missing values.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value == "":
        return np.nan

    return value


def clean_text_columns(df, text_columns):
    """
    Applies clean_text() to selected text columns if they exist.
    """
    df = df.copy()

    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

    return df


def is_blank_or_unknown(value):
    """
    Flags blank, missing, or placeholder values.
    """
    if pd.isna(value):
        return True

    value = str(value).strip().lower()

    return value in [
        "", "na", "n/a", "none", "null", "unknown",
        "missing", "not available", "-"
    ]


def vague_description(value, min_words=8):
    """
    Flags incident descriptions that are too short or too generic.
    """
    if is_blank_or_unknown(value):
        return True

    text = str(value).strip().lower()
    words = re.findall(r"\b\w+\b", text)

    vague_terms = [
        "issue", "problem", "fault", "damaged",
        "broken", "bad", "defective", "incorrect",
        "quality issue", "not working"
    ]

    if len(words) < min_words:
        return True

    if text in vague_terms:
        return True

    return False


def standardise_supplier_name(name):
    """
    Creates a clean supplier name for grouping and reporting.

    This version preserves the original capitalisation of supplier names,
    including branded names such as 'StratoSpacer Technologies'.
    """
    if pd.isna(name):
        return np.nan

    # Strip leading/trailing spaces and collapse repeated spaces
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)

    # Manual corrections can be added here if needed
    supplier_mapping = {
        # Example corrections:
        # "stratospacer technologies": "StratoSpacer Technologies",
        # "strato spacer technologies": "StratoSpacer Technologies",
        # "velcronix ltd": "Velcronix",
    }

    # Use lowercase version only for matching against manual correction keys
    name_key = name.lower()

    return supplier_mapping.get(name_key, name)


def normalise_supplier_name_for_matching(name):
    """
    Creates a simplified supplier name for detecting inconsistent naming.
    """
    if pd.isna(name):
        return np.nan

    name = str(name).lower().strip()
    name = re.sub(r"[^a-z0-9\s]", "", name)

    remove_words = [
        "ltd", "limited", "inc", "corp", "corporation",
        "co", "company", "llc", "plc", "gmbh"
    ]

    words = [word for word in name.split() if word not in remove_words]
    return " ".join(words)


def similar(a, b):
    """
    Calculates similarity between two supplier names.
    """
    return SequenceMatcher(None, str(a), str(b)).ratio()


def add_percentage_column(
    df,
    count_col,
    total_count,
    new_col="percentage_of_total_incidents"
):
    """
    Adds a rounded percentage column to a summary table.
    """
    df = df.copy()
    df[new_col] = (df[count_col] / total_count * 100).round(2)
    return df


def create_count_summary(df, group_col, total_count, count_col="incident_count"):
    """
    Creates a count and percentage summary for one grouping column.
    """
    summary = (
        df.groupby(group_col, dropna=False)
          .size()
          .reset_index(name=count_col)
          .sort_values(count_col, ascending=False)
    )

    summary = add_percentage_column(
        summary,
        count_col=count_col,
        total_count=total_count
    )

    return summary


def export_tables_to_excel(output_file, tables):
    """
    Exports multiple DataFrames to an Excel workbook.

    Parameters
    ----------
    output_file : str
        Name of the output Excel file.

    tables : dict
        Dictionary where keys are sheet names and values are DataFrames.
    """
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sheet_name, table in tables.items():
            table.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"Tables exported to: {output_file}")

In [75]:
# ============================================================
# LOAD DATA
# ============================================================

df_raw = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

# Keep original Excel row number for easier review
df_raw["excel_row"] = df_raw.index + 2

# Clean column names immediately so the whole notebook uses one naming style
df = clean_column_names(df_raw)

print(df.head())
print(df.info())

     part_number cage_code manufacturer          description serial_number  \
0  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   
1  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   
2  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   
3  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   
4  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   

   qty incident_type incident_date incident_status incident_closed_date  \
0  1.0           RMA    2025-04-03          Closed           2025-04-18   
1  3.0           RMA    2025-04-03          Closed           2025-04-18   
2  2.0           RMA    2025-04-03          Closed           2025-04-18   
3  1.0           RMA    2025-04-03          Closed           2025-04-18   
4  1.0           RMA    2025-04-03          Closed           2025-04-18   

   duration_days                               incident_description  \
0        

In [76]:
# ============================================================
# BASIC DATA CLEANING
# ============================================================

# Remove rows that are completely empty
df = df.dropna(how="all").copy()

# ------------------------------------------------------------
# Remove non-incident summary rows
# ------------------------------------------------------------

# These fields should exist for a real incident record.
# A row with only an average/summary value and no operational identifiers
# should not be treated as an incident.
required_incident_fields = [
    "part_number",
    "cage_code",
    "manufacturer",
    "incident_type",
    "incident_date",
    "incident_description"
]

existing_required_fields = [
    col for col in required_incident_fields
    if col in df.columns
]

# Count how many real incident-identifying fields are present per row
df["incident_field_count"] = df[existing_required_fields].notna().sum(axis=1)

# A likely summary/non-incident row has no real incident-identifying fields
df["likely_summary_row"] = df["incident_field_count"] == 0

likely_summary_rows = df[df["likely_summary_row"]].copy()

# Keep only actual incident rows for analysis
df = df[~df["likely_summary_row"]].copy()

# Optional: remove helper column after filtering
df = df.drop(columns=["incident_field_count"])

# Standardise key text columns
text_columns = [
    "part_number",
    "cage_code",
    "manufacturer",
    "description",
    "serial_number",
    "incident_type",
    "incident_status",
    "incident_description"
]

df = clean_text_columns(df, text_columns)

# Convert quantity and cost fields to numeric values
if "qty" in df.columns:
    df["qty"] = pd.to_numeric(df["qty"], errors="coerce")

if "resolution_cost" in df.columns:
    df["resolution_cost_original"] = df["resolution_cost"]
    df["resolution_cost"] = pd.to_numeric(df["resolution_cost"], errors="coerce")

# Convert date fields
date_columns = ["incident_date", "incident_closed_date"]

for col in date_columns:
    if col in df.columns:
        df[col + "_original"] = df[col]
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Standardise supplier names
df["supplier_clean"] = df["manufacturer"].apply(standardise_supplier_name)

# Standardise incident type values
incident_type_mapping = {
    "rma": "RMA",
    "work order": "Work Order"
}

df["incident_type_clean"] = (
    df["incident_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .map(incident_type_mapping)
    .fillna(df["incident_type"])
)

# Calculate incident duration
df["calculated_duration_days"] = (
    df["incident_closed_date"] - df["incident_date"]
).dt.days

# For open incidents, calculate current age
today = pd.Timestamp.today().normalize()

df["open_incident_age_days"] = np.where(
    df["incident_closed_date"].isna(),
    (today - df["incident_date"]).dt.days,
    np.nan
)

# Use calculated duration for closed incidents
df["resolution_duration_days"] = df["calculated_duration_days"]

# Compare calculated duration to existing duration column if it exists
if "duration_days" in df.columns:
    df["duration_difference"] = (
        df["duration_days"] - df["calculated_duration_days"]
    )

print(df.head())
print(df.info())

     part_number cage_code manufacturer          description serial_number  \
0  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   
1  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   
2  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   
3  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   
4  M39029/63-368     81349    Velcronix  Contact, Electrical           NaN   

   qty incident_type incident_date incident_status incident_closed_date  ...  \
0  1.0           RMA    2025-04-03          Closed           2025-04-18  ...   
1  3.0           RMA    2025-04-03          Closed           2025-04-18  ...   
2  2.0           RMA    2025-04-03          Closed           2025-04-18  ...   
3  1.0           RMA    2025-04-03          Closed           2025-04-18  ...   
4  1.0           RMA    2025-04-03          Closed           2025-04-18  ...   

   likely_summary_row resolution_cost_original  in

In [77]:
# ============================================================
# DATA QUALITY CHECKS
# ============================================================

# ------------------------------------------------------------
# 1. Missing supplier names
# ------------------------------------------------------------

df["flag_missing_supplier"] = df["manufacturer"].apply(is_blank_or_unknown)


# ------------------------------------------------------------
# 2. Inconsistent supplier naming
# ------------------------------------------------------------

df["supplier_normalised"] = df["manufacturer"].apply(
    normalise_supplier_name_for_matching
)

supplier_variants = (
    df.dropna(subset=["manufacturer"])
      .groupby("supplier_normalised")["manufacturer"]
      .nunique()
      .reset_index(name="supplier_name_variants")
)

# Prevent duplicate merge columns if this cell is rerun
df = df.drop(
    columns=[col for col in df.columns if col.startswith("supplier_name_variants")],
    errors="ignore"
)

df = df.merge(
    supplier_variants,
    on="supplier_normalised",
    how="left"
)

df["flag_supplier_name_variant"] = df["supplier_name_variants"] > 1


# Optional fuzzy matching between supplier names
unique_suppliers = (
    df["manufacturer"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

fuzzy_supplier_matches = []

for i, supplier_a in enumerate(unique_suppliers):
    for supplier_b in unique_suppliers[i + 1:]:
        score = similar(
            normalise_supplier_name_for_matching(supplier_a),
            normalise_supplier_name_for_matching(supplier_b)
        )

        if score >= 0.80 and supplier_a != supplier_b:
            fuzzy_supplier_matches.append({
                "supplier_a": supplier_a,
                "supplier_b": supplier_b,
                "similarity_score": round(score, 3)
            })

fuzzy_supplier_matches = pd.DataFrame(fuzzy_supplier_matches)


# ------------------------------------------------------------
# 3. Missing or invalid incident types
# ------------------------------------------------------------

df["flag_missing_incident_type"] = df["incident_type"].apply(is_blank_or_unknown)

df["flag_invalid_incident_type"] = ~df["incident_type_clean"].isin(
    VALID_INCIDENT_TYPES
)

df.loc[
    df["flag_missing_incident_type"],
    "flag_invalid_incident_type"
] = True


# ------------------------------------------------------------
# 4. Missing, negative, unusual, or high resolution costs
# ------------------------------------------------------------

df["flag_missing_resolution_cost"] = df["resolution_cost"].isna()

df["flag_negative_resolution_cost"] = df["resolution_cost"] < 0


# IQR-based high-cost indicator
# This is treated as a business indicator, not a data quality issue.
costs = df["resolution_cost"].dropna()

if len(costs) > 0:
    q1 = costs.quantile(0.25)
    q3 = costs.quantile(0.75)
    iqr = q3 - q1
    upper_cost_limit = q3 + 1.5 * iqr
else:
    upper_cost_limit = np.nan

df["business_unusually_high_resolution_cost_iqr"] = (
    df["resolution_cost"] > upper_cost_limit
)


# Business threshold check
# This is NOT a data quality issue.
# It is a business indicator showing high-cost incidents.
df["business_high_resolution_cost"] = (
    df["resolution_cost"] > BUSINESS_COST_THRESHOLD
)


# Optional separate table for high-cost incidents
# This is useful for management reporting but should not feed into
# the data quality review queue.
high_cost_incidents = df[
    df["business_high_resolution_cost"]
    | df["business_unusually_high_resolution_cost_iqr"]
].copy()


# ------------------------------------------------------------
# 5. Missing, invalid, or inconsistent date fields
# ------------------------------------------------------------

df["flag_missing_incident_date"] = df["incident_date"].isna()

df["flag_missing_closed_date_when_closed"] = (
    df["incident_status"].astype(str).str.lower().eq("closed")
    & df["incident_closed_date"].isna()
)

df["flag_closed_date_before_incident_date"] = (
    df["incident_closed_date"].notna()
    & df["incident_date"].notna()
    & (df["incident_closed_date"] < df["incident_date"])
)

df["flag_open_incident_with_closed_date"] = (
    df["incident_status"].astype(str).str.lower().eq("open")
    & df["incident_closed_date"].notna()
)

if "duration_days" in df.columns:
    df["flag_duration_mismatch"] = (
        df["duration_days"].notna()
        & df["calculated_duration_days"].notna()
        & (df["duration_days"] != df["calculated_duration_days"])
    )
else:
    df["flag_duration_mismatch"] = False


# ------------------------------------------------------------
# 6. Duplicate incident records
# ------------------------------------------------------------

duplicate_check_columns = [
    "part_number",
    "cage_code",
    "manufacturer",
    "description",
    "serial_number",
    "qty",
    "incident_type",
    "incident_date",
    "incident_closed_date",
    "incident_description",
    "resolution_cost"
]

duplicate_check_columns = [
    col for col in duplicate_check_columns if col in df.columns
]

df["flag_duplicate_record"] = df.duplicated(
    subset=duplicate_check_columns,
    keep=False
)

near_duplicate_columns = [
    "part_number",
    "cage_code",
    "manufacturer",
    "description",
    "incident_type",
    "incident_date",
    "incident_closed_date",
    "incident_description",
    "resolution_cost"
]

near_duplicate_columns = [
    col for col in near_duplicate_columns if col in df.columns
]

df["flag_near_duplicate_no_serial"] = (
    df["serial_number"].isna()
    & df.duplicated(subset=near_duplicate_columns, keep=False)
)


# ------------------------------------------------------------
# 7. Blank or vague descriptions
# ------------------------------------------------------------

df["flag_blank_incident_description"] = (
    df["incident_description"].apply(is_blank_or_unknown)
)

df["flag_vague_incident_description"] = (
    df["incident_description"].apply(vague_description)
)


# ------------------------------------------------------------
# 8. Overall review flag for data quality
# ------------------------------------------------------------

# Only columns starting with "flag_" are treated as data quality issues.
# Business indicators are intentionally excluded.
flag_columns = [col for col in df.columns if col.startswith("flag_")]

df["total_data_quality_flags"] = df[flag_columns].sum(axis=1)

df["needs_data_quality_review"] = df["total_data_quality_flags"] > 0


# ------------------------------------------------------------
# 9. Create issue-specific review tables
# ------------------------------------------------------------

missing_suppliers = df[df["flag_missing_supplier"]].copy()

supplier_name_issues = df[df["flag_supplier_name_variant"]].copy()

incident_type_issues = df[
    df["flag_missing_incident_type"]
    | df["flag_invalid_incident_type"]
].copy()

# Cost issues only include actual data quality concerns.
# High or unusually high costs are handled separately in high_cost_incidents.
cost_issues = df[
    df["flag_missing_resolution_cost"]
    | df["flag_negative_resolution_cost"]
].copy()

date_issues = df[
    df["flag_missing_incident_date"]
    | df["flag_missing_closed_date_when_closed"]
    | df["flag_closed_date_before_incident_date"]
    | df["flag_open_incident_with_closed_date"]
    | df["flag_duration_mismatch"]
].copy()

duplicate_issues = df[
    df["flag_duplicate_record"]
    | df["flag_near_duplicate_no_serial"]
].copy()

description_issues = df[
    df["flag_blank_incident_description"]
    | df["flag_vague_incident_description"]
].copy()

all_flagged_records = df[df["needs_data_quality_review"]].copy()


# ------------------------------------------------------------
# 10. Data quality and business indicator summary table
# ------------------------------------------------------------

data_quality_summary = pd.DataFrame({
    "check_or_indicator": [
        "Likely non-incident summary rows removed",
        "Missing supplier",
        "Supplier name variants",
        "Fuzzy supplier name matches",
        "Missing incident type",
        "Invalid incident type",
        "Missing resolution cost",
        "Negative resolution cost",
        "Unusually high resolution cost using IQR",
        "High resolution cost above business threshold",
        "Missing incident date",
        "Missing closed date when status is closed",
        "Closed date before incident date",
        "Open incident with closed date",
        "Duration mismatch",
        "Duplicate records",
        "Near duplicate records without serial number",
        "Blank incident description",
        "Vague incident description",
        "Records needing data quality review"
    ],
    "issue_count": [
        len(likely_summary_rows),
        df["flag_missing_supplier"].sum(),
        df["flag_supplier_name_variant"].sum(),
        len(fuzzy_supplier_matches),
        df["flag_missing_incident_type"].sum(),
        df["flag_invalid_incident_type"].sum(),
        df["flag_missing_resolution_cost"].sum(),
        df["flag_negative_resolution_cost"].sum(),
        df["business_unusually_high_resolution_cost_iqr"].sum(),
        df["business_high_resolution_cost"].sum(),
        df["flag_missing_incident_date"].sum(),
        df["flag_missing_closed_date_when_closed"].sum(),
        df["flag_closed_date_before_incident_date"].sum(),
        df["flag_open_incident_with_closed_date"].sum(),
        df["flag_duration_mismatch"].sum(),
        df["flag_duplicate_record"].sum(),
        df["flag_near_duplicate_no_serial"].sum(),
        df["flag_blank_incident_description"].sum(),
        df["flag_vague_incident_description"].sum(),
        df["needs_data_quality_review"].sum()
    ]
})

data_quality_summary["percentage_of_incident_records"] = (
    data_quality_summary["issue_count"] / len(df) * 100
).round(2)

data_quality_summary

,check_or_indicator,issue_count,percentage_of_incident_records
0,Likely non-incident summary rows removed,2,4.0
1,Missing supplier,0,0.0
2,Supplier name variants,0,0.0
3,Fuzzy supplier name matches,0,0.0
4,Missing incident type,0,0.0
5,Invalid incident type,0,0.0
6,Missing resolution cost,10,20.0
7,Negative resolution cost,0,0.0
8,Unusually high resolution cost using IQR,9,18.0
9,High resolution cost above business threshold,1,2.0


In [78]:
# ============================================================
# QUANTITATIVE ANALYSIS TABLES
# ============================================================

total_incidents = len(df)

# ------------------------------------------------------------
# Incidents per supplier
# ------------------------------------------------------------

incidents_per_supplier = (
    df.dropna(subset=["supplier_clean"])
      .groupby("supplier_clean")
      .size()
      .reset_index(name="number_of_incidents")
      .sort_values("number_of_incidents", ascending=False)
)

incidents_per_supplier = add_percentage_column(
    incidents_per_supplier,
    count_col="number_of_incidents",
    total_count=total_incidents
)

incidents_per_supplier


# ------------------------------------------------------------
# Resolution cost
# ------------------------------------------------------------

total_resolution_cost = df["resolution_cost"].sum(skipna=True)

print(f"Total resolution cost: {total_resolution_cost:,.2f}")


# ------------------------------------------------------------
# Average resolution duration
# ------------------------------------------------------------

average_resolution_duration = df["resolution_duration_days"].mean(skipna=True)

print(f"Average resolution duration: {average_resolution_duration:.2f} days")


# ------------------------------------------------------------
# Duration statistics by incident type
# ------------------------------------------------------------

duration_stats_by_type = (
    df.dropna(subset=["incident_type_clean"])
      .groupby("incident_type_clean")["resolution_duration_days"]
      .agg(
          mean_duration="mean",
          median_duration="median",
          min_duration="min",
          max_duration="max",
          incident_count="count"
      )
      .reset_index()
)

duration_stats_by_type = duration_stats_by_type.round({
    "mean_duration": 2,
    "median_duration": 2,
    "min_duration": 2,
    "max_duration": 2
})

duration_stats_by_type


# ------------------------------------------------------------
# Supplier summary
# ------------------------------------------------------------

supplier_summary = (
    df.dropna(subset=["supplier_clean"])
      .groupby("supplier_clean")
      .agg(
          number_of_incidents=("supplier_clean", "size"),
          total_resolution_cost=("resolution_cost", "sum"),
          average_resolution_cost=("resolution_cost", "mean"),
          average_resolution_duration_days=("resolution_duration_days", "mean"),
          median_resolution_duration_days=("resolution_duration_days", "median"),
          min_resolution_duration_days=("resolution_duration_days", "min"),
          max_resolution_duration_days=("resolution_duration_days", "max")
      )
      .reset_index()
)

supplier_summary = add_percentage_column(
    supplier_summary,
    count_col="number_of_incidents",
    total_count=total_incidents
)

supplier_summary = supplier_summary.round({
    "total_resolution_cost": 2,
    "average_resolution_cost": 2,
    "average_resolution_duration_days": 2,
    "median_resolution_duration_days": 2,
    "min_resolution_duration_days": 2,
    "max_resolution_duration_days": 2,
    "percentage_of_total_incidents": 2
})

supplier_summary = supplier_summary.sort_values(
    "number_of_incidents",
    ascending=False
)

supplier_summary


# ------------------------------------------------------------
# Overall KPI summary
# ------------------------------------------------------------

overall_summary = pd.DataFrame({
    "metric": [
        "Total incidents",
        "Total resolution cost",
        "Average resolution duration days",
        "Number of suppliers",
        "Number of RMAs",
        "Number of work orders"
    ],
    "value": [
        total_incidents,
        round(total_resolution_cost, 2),
        round(average_resolution_duration, 2),
        df["supplier_clean"].nunique(dropna=True),
        (df["incident_type_clean"] == "RMA").sum(),
        (df["incident_type_clean"] == "Work Order").sum()
    ]
})


# ------------------------------------------------------------
# Incident type summary
# ------------------------------------------------------------

incident_type_summary = (
    df.dropna(subset=["incident_type_clean"])
      .groupby("incident_type_clean")
      .agg(
          number_of_incidents=("incident_type_clean", "size"),
          total_resolution_cost=("resolution_cost", "sum"),
          average_resolution_cost=("resolution_cost", "mean"),
          mean_duration_days=("resolution_duration_days", "mean"),
          median_duration_days=("resolution_duration_days", "median"),
          min_duration_days=("resolution_duration_days", "min"),
          max_duration_days=("resolution_duration_days", "max")
      )
      .reset_index()
)

incident_type_summary = add_percentage_column(
    incident_type_summary,
    count_col="number_of_incidents",
    total_count=total_incidents
)

incident_type_summary = incident_type_summary.round({
    "total_resolution_cost": 2,
    "average_resolution_cost": 2,
    "mean_duration_days": 2,
    "median_duration_days": 2,
    "min_duration_days": 2,
    "max_duration_days": 2,
    "percentage_of_total_incidents": 2
})

incident_type_summary = incident_type_summary.sort_values(
    "number_of_incidents",
    ascending=False
)

incident_type_summary

Total resolution cost: 82,567.00
Average resolution duration: 23.02 days


,incident_type_clean,number_of_incidents,total_resolution_cost,average_resolution_cost,mean_duration_days,median_duration_days,min_duration_days,max_duration_days,percentage_of_total_incidents
0,RMA,25,4250.0,283.33,26.32,16.0,4,65,50.0
1,Work Order,25,78317.0,3132.68,19.72,15.0,2,61,50.0


In [79]:
# ============================================================
# CHART-READY TABLES FOR POWERPOINT
# ============================================================

# Chart 1: incidents by supplier
chart_incidents_by_supplier = incidents_per_supplier.copy()

# Chart 2: percentage of total incidents by supplier
chart_supplier_percentage = incidents_per_supplier[[
    "supplier_clean",
    "percentage_of_total_incidents"
]].copy()

# Chart 3: total resolution cost by supplier
chart_cost_by_supplier = (
    supplier_summary[[
        "supplier_clean",
        "total_resolution_cost"
    ]]
    .sort_values("total_resolution_cost", ascending=False)
)

# Chart 4: average resolution duration by supplier
chart_duration_by_supplier = (
    supplier_summary[[
        "supplier_clean",
        "average_resolution_duration_days"
    ]]
    .sort_values("average_resolution_duration_days", ascending=False)
)

# Chart 5: duration statistics for RMAs vs work orders
chart_duration_by_incident_type = duration_stats_by_type.copy()

chart_incidents_by_supplier

,supplier_clean,number_of_incidents,percentage_of_total_incidents
1,Aperion Defense,10,20.0
7,Orionist Systems,9,18.0
10,Velcronix,6,12.0
0,AeroNovus Instruments,5,10.0
11,Viryonox,4,8.0
2,Booftie Technologies,4,8.0
6,Morogdon Defense,4,8.0
4,Graveson-Reed,4,8.0
3,CommandOne Defense,1,2.0
5,KinextLink Communications,1,2.0


In [80]:
# ============================================================
# EXPORT QUANTITATIVE SUMMARY TABLES TO EXCEL
# ============================================================

summary_tables = {
    "Overall Summary": overall_summary,
    "Supplier Summary": supplier_summary,
    "Incident Type Summary": incident_type_summary,
    "Duration Stats": duration_stats_by_type,
    "Chart Incidents Supplier": chart_incidents_by_supplier,
    "Chart Supplier Percent": chart_supplier_percentage,
    "Chart Cost Supplier": chart_cost_by_supplier,
    "Chart Duration Supplier": chart_duration_by_supplier,
    "Chart Duration Type": chart_duration_by_incident_type,
    "Data Quality Summary": data_quality_summary
}

export_tables_to_excel(
    output_file="incident_analysis_summary_tables.xlsx",
    tables=summary_tables
)

Tables exported to: incident_analysis_summary_tables.xlsx


In [81]:
# ============================================================
# GEMINI API SETUP
# ============================================================

!pip install -q google-genai

from google import genai
from google.genai import types
from google.colab import userdata

# Store your Gemini API key in Colab:
# Left sidebar > Secrets > Add new secret > Name: GEMINI_API_KEY
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if GEMINI_API_KEY is None:
    raise ValueError("Gemini API key not found. Add GEMINI_API_KEY to Colab Secrets.")

client = genai.Client(api_key=GEMINI_API_KEY)

In [82]:
# ============================================================
# ROOT-CAUSE CLASSIFICATION SCHEMA
# ============================================================

ROOT_CAUSE_CATEGORIES = [
    "Damaged packaging",
    "Defective product or part",
    "Incorrect item shipped",
    "Incorrect quantity shipped",
    "Poor product quality",
    "Material damaged during warehouse handling",
    "Documentation or labelling issue",
    "Supplier delivery issue",
    "Internal warehouse process issue",
    "Other",
    "Unclear / insufficient detail"
]

root_cause_schema = {
    "type": "object",
    "properties": {
        "root_cause_category": {
            "type": "string",
            "enum": ROOT_CAUSE_CATEGORIES
        },
        "confidence": {
            "type": "string",
            "enum": ["high", "medium", "low"]
        },
        "justification": {
            "type": "string"
        }
    },
    "required": [
        "root_cause_category",
        "confidence",
        "justification"
    ]
}

In [83]:
# ============================================================
# GEMINI ROOT-CAUSE CLASSIFICATION FUNCTION
# ============================================================

def classify_incident_with_gemini(row, model="gemini-2.5-flash"):
    """
    Sends one incident description to Gemini and returns:
    - root cause category
    - confidence
    - short justification

    The model is instructed not to guess and to use only the incident description.
    """

    incident_id = row.get("incident_id", row.name)
    incident_type = row.get("incident_type_clean", row.get("incident_type", ""))
    supplier = row.get("supplier_clean", row.get("manufacturer", ""))
    description = row.get("incident_description", "")

    prompt = f"""
You are an inventory incident root-cause analyst.

Classify the incident description into ONE primary root-cause category.

Use only the wording in the description. Do not invent facts.

If the description is unclear, vague, or does not provide enough information,
classify it as "Unclear / insufficient detail".

Suggested categories:
- Damaged packaging
- Defective product or part
- Incorrect item shipped
- Incorrect quantity shipped
- Poor product quality
- Material damaged during warehouse handling
- Documentation or labelling issue
- Supplier delivery issue
- Internal warehouse process issue
- Other
- Unclear / insufficient detail

Incident ID: {incident_id}
Incident Type: {incident_type}
Supplier: {supplier}
Description: {description}

Return JSON with:
- root_cause_category
- confidence
- justification
"""

    try:
        response = client.models.generate_content(
            model=model,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=root_cause_schema,
                temperature=0
            )
        )

        result = json.loads(response.text)

        return {
            "root_cause_category": result.get(
                "root_cause_category",
                "Unclear / insufficient detail"
            ),
            "root_cause_confidence": result.get("confidence", "low"),
            "root_cause_justification": result.get(
                "justification",
                "No justification returned."
            )
        }

    except Exception as e:
        return {
            "root_cause_category": "Unclear / insufficient detail",
            "root_cause_confidence": "low",
            "root_cause_justification": f"Gemini classification failed: {e}"
        }

In [84]:
# ============================================================
# PREPARE INCIDENTS FOR LLM CLASSIFICATION
# ============================================================

# Create incident_id if the dataset does not already have one
if "incident_id" not in df.columns:
    df["incident_id"] = "INC_" + (df.index + 1).astype(str)

# Make sure incident descriptions are strings
df["incident_description"] = df["incident_description"].fillna("").astype(str)

# Classify only rows with a non-empty description
df_to_classify = df[df["incident_description"].str.strip() != ""].copy()

print(f"Number of incidents to classify: {len(df_to_classify)}")

Number of incidents to classify: 50


In [85]:
# ============================================================
# RUN GEMINI CLASSIFICATION
# ============================================================

classification_results = []

for idx, row in tqdm(df_to_classify.iterrows(), total=len(df_to_classify)):
    result = classify_incident_with_gemini(row)
    result["index"] = idx
    classification_results.append(result)

    # Small pause to reduce rate-limit risk
    time.sleep(0.2)

if classification_results:
    classification_df = pd.DataFrame(classification_results).set_index("index")
else:
    classification_df = pd.DataFrame(
        columns=[
            "root_cause_category",
            "root_cause_confidence",
            "root_cause_justification"
        ]
    )


# ------------------------------------------------------------
# Drop previous classification columns if this cell is rerun
# ------------------------------------------------------------

classification_cols = [
    "root_cause_category",
    "root_cause_confidence",
    "root_cause_justification",
    "root_cause_category_original_gemini",
    "root_cause_confidence_original_gemini",
    "root_cause_justification_original_gemini",
    "root_cause_used_fallback",
    "review_needed"
]

df = df.drop(columns=[col for col in classification_cols if col in df.columns])


# ------------------------------------------------------------
# Join Gemini classification results back to main dataframe
# ------------------------------------------------------------

df = df.join(classification_df)


# ------------------------------------------------------------
# Fill rows that were not classified because description was blank
# ------------------------------------------------------------

df["root_cause_category"] = df["root_cause_category"].fillna(
    "Unclear / insufficient detail"
)

df["root_cause_confidence"] = df["root_cause_confidence"].fillna("low")

df["root_cause_justification"] = df["root_cause_justification"].fillna(
    "Blank or missing description"
)

df[[
    "incident_id",
    "incident_description",
    "root_cause_category",
    "root_cause_confidence",
    "root_cause_justification"
]].head()

100%|██████████| 50/50 [00:24<00:00,  2.05it/s]


,incident_id,incident_description,root_cause_category,root_cause_confidence,root_cause_justification
0,INC_1,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,low,Gemini classification failed: 429 RESOURCE_EXH...
1,INC_2,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,low,Gemini classification failed: 429 RESOURCE_EXH...
2,INC_3,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,low,Gemini classification failed: 429 RESOURCE_EXH...
3,INC_4,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,low,Gemini classification failed: 429 RESOURCE_EXH...
4,INC_5,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,low,Gemini classification failed: 429 RESOURCE_EXH...


In [86]:
# ============================================================
# RULE-BASED FALLBACK FOR UNCLEAR GEMINI CLASSIFICATIONS
# ============================================================

def rule_based_root_cause(description):
    """
    Transparent fallback classifier for repeated incident-description patterns.

    This is only used where Gemini returned 'Unclear / insufficient detail'.
    It improves consistency for clear recurring phrases in the dataset.
    """

    text = str(description).lower().strip()

    if text == "":
        return "Unclear / insufficient detail", "low", "Blank or missing description"

    # --------------------------------------------------------
    # Material damaged during warehouse handling
    # --------------------------------------------------------
    if any(term in text for term in [
        "dropped from",
        "forklift",
        "incorrect forklift",
        "damaged due to incorrect forklift",
        "issued from warehouse were damaged"
    ]):
        return (
            "Material damaged during warehouse handling",
            "high",
            "Description mentions warehouse handling damage such as a dropped item or incorrect forklift usage."
        )

    # --------------------------------------------------------
    # Internal warehouse process issue
    # --------------------------------------------------------
    if any(term in text for term in [
        "incorrectly packaged for esd",
        "esd sensitivity",
        "upon issuing circuit cards from warehouse",
        "engineering will be rotating stock",
        "rotating stock to the lab",
        "screws were issued from warehouse",
        "warehouse shelves"
    ]):
        return (
            "Internal warehouse process issue",
            "high",
            "Description mentions a warehouse process issue such as ESD packaging, stock rotation, or warehouse-issued parts."
        )

    # --------------------------------------------------------
    # Documentation or labelling issue
    # --------------------------------------------------------
    if any(term in text for term in [
        "incorrectly labeled",
        "incorrectly labelled",
        "new labels",
        "packing slips",
        "correct part number on units and packing slips"
    ]):
        return (
            "Documentation or labelling issue",
            "high",
            "Description mentions incorrect labels, packing slips, or part-number documentation."
        )

    # --------------------------------------------------------
    # Incorrect item shipped
    # --------------------------------------------------------
    if any(term in text for term in [
        "incorrect item received",
        "incorrect item",
        "incorrect part number"
    ]):
        return (
            "Incorrect item shipped",
            "high",
            "Description mentions an incorrect item or incorrect part number."
        )

    # --------------------------------------------------------
    # Incorrect quantity shipped
    # --------------------------------------------------------
    if any(term in text for term in [
        "incorrect qty",
        "incorrect quantity",
        "missing quantity",
        "did not have all required screws",
        "required screws"
    ]):
        return (
            "Incorrect quantity shipped",
            "high",
            "Description mentions incorrect quantity or missing required screws."
        )

    # --------------------------------------------------------
    # Defective product or part
    # --------------------------------------------------------
    if any(term in text for term in [
        "cracks",
        "weld",
        "mechanical wear",
        "unit failure",
        "out of tolerance",
        "incorrect firmware",
        "software loads",
        "not working",
        "incapable of performing"
    ]):
        return (
            "Defective product or part",
            "high",
            "Description mentions defect evidence such as cracks, weld issues, mechanical wear, unit failure, out of tolerance, or firmware/software issues."
        )

    # --------------------------------------------------------
    # Poor product quality / non-compliance
    # --------------------------------------------------------
    if any(term in text for term in [
        "non compliant",
        "not compliant",
        "non-compliant",
        "failed quality",
        "failed quality check",
        "failed quality inspection",
        "did not pass quality",
        "quality inspection",
        "quality check",
        "mil-std",
        "non-compliant paint"
    ]):
        return (
            "Poor product quality",
            "high",
            "Description mentions non-compliance, MIL-STD, failed quality check, or quality inspection failure."
        )

    # --------------------------------------------------------
    # Supplier delivery issue
    # --------------------------------------------------------
    if any(term in text for term in [
        "supply chain delays",
        "replacement units not received until",
        "delayed replacement",
        "not received until"
    ]):
        return (
            "Supplier delivery issue",
            "high",
            "Description mentions supply chain delays or delayed replacement units."
        )

    # --------------------------------------------------------
    # Damaged packaging
    # --------------------------------------------------------
    if any(term in text for term in [
        "waterstained packaging",
        "waterstained",
        "damaged packaging",
        "damaged box",
        "crushed box",
        "torn packaging",
        "box damaged"
    ]):
        return (
            "Damaged packaging",
            "medium",
            "Description mentions packaging, box damage, or waterstaining."
        )

    return (
        "Unclear / insufficient detail",
        "low",
        "No clear root-cause phrase matched."
    )


# ------------------------------------------------------------
# Keep original Gemini output for auditability
# ------------------------------------------------------------

df["root_cause_category_original_gemini"] = df["root_cause_category"]
df["root_cause_confidence_original_gemini"] = df["root_cause_confidence"]
df["root_cause_justification_original_gemini"] = df["root_cause_justification"]


# ------------------------------------------------------------
# Apply fallback only where Gemini returned unclear
# ------------------------------------------------------------

fallback_results = df.apply(
    lambda row: rule_based_root_cause(row["incident_description"])
    if row["root_cause_category"] == "Unclear / insufficient detail"
    else (
        row["root_cause_category"],
        row["root_cause_confidence"],
        row["root_cause_justification"]
    ),
    axis=1
)

df["root_cause_category"] = fallback_results.apply(lambda x: x[0])
df["root_cause_confidence"] = fallback_results.apply(lambda x: x[1])
df["root_cause_justification"] = fallback_results.apply(lambda x: x[2])


# ------------------------------------------------------------
# Identify where fallback changed Gemini's original classification
# ------------------------------------------------------------

df["root_cause_used_fallback"] = (
    df["root_cause_category_original_gemini"] == "Unclear / insufficient detail"
) & (
    df["root_cause_category"] != "Unclear / insufficient detail"
)

df[[
    "incident_id",
    "incident_description",
    "root_cause_category_original_gemini",
    "root_cause_category",
    "root_cause_confidence",
    "root_cause_used_fallback"
]].head()

,incident_id,incident_description,root_cause_category_original_gemini,root_cause_category,root_cause_confidence,root_cause_used_fallback
0,INC_1,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True
1,INC_2,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True
2,INC_3,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True
3,INC_4,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True
4,INC_5,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True


In [87]:
# ============================================================
# HUMAN REVIEW FLAGGING
# ============================================================

def assign_review_needed(row):
    """
    Flags incidents that should be manually reviewed by a human.

    Fallback use alone does not automatically require review because some
    fallback rules are clear and auditable.
    """

    category = row["root_cause_category"]
    confidence = str(row["root_cause_confidence"]).lower()
    used_fallback = row.get("root_cause_used_fallback", False)

    # Always review unclear classifications
    if category == "Unclear / insufficient detail":
        return "Yes"

    # Always review low-confidence classifications
    if confidence == "low":
        return "Yes"

    # Review fallback classifications only where confidence is not high
    if used_fallback and confidence in ["medium", "low"]:
        return "Yes"

    # Review categories that are often ambiguous
    if category in [
        "Damaged packaging",
        "Documentation or labelling issue",
        "Incorrect quantity shipped"
    ]:
        return "Yes"

    return "No"


df["review_needed"] = df.apply(assign_review_needed, axis=1)


classification_review_table = df[[
    "incident_id",
    "incident_type_clean",
    "supplier_clean",
    "incident_description",
    "root_cause_category_original_gemini",
    "root_cause_category",
    "root_cause_used_fallback",
    "root_cause_confidence",
    "review_needed",
    "root_cause_justification"
]].copy()

classification_review_table.head(20)

,incident_id,incident_type_clean,supplier_clean,incident_description,root_cause_category_original_gemini,root_cause_category,root_cause_used_fallback,root_cause_confidence,review_needed,root_cause_justification
0,INC_1,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,True,high,No,"Description mentions non-compliance, MIL-STD, ..."
1,INC_2,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,True,high,No,"Description mentions non-compliance, MIL-STD, ..."
2,INC_3,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,True,high,No,"Description mentions non-compliance, MIL-STD, ..."
3,INC_4,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,True,high,No,"Description mentions non-compliance, MIL-STD, ..."
4,INC_5,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,True,high,No,"Description mentions non-compliance, MIL-STD, ..."
5,INC_6,RMA,Velcronix,Item received was not compliant with MIL-STD a...,Unclear / insufficient detail,Poor product quality,True,high,No,"Description mentions non-compliance, MIL-STD, ..."
6,INC_7,RMA,Aperion Defense,Item failed quality inspection. Box door weld ...,Unclear / insufficient detail,Defective product or part,True,high,No,Description mentions defect evidence such as c...
7,INC_8,Work Order,Aperion Defense,Mounting plates did not have all required scre...,Unclear / insufficient detail,Internal warehouse process issue,True,high,No,Description mentions a warehouse process issue...
8,INC_9,RMA,Aperion Defense,Item failed quality inspection. Box door weld ...,Unclear / insufficient detail,Defective product or part,True,high,No,Description mentions defect evidence such as c...
9,INC_10,RMA,Aperion Defense,Item failed quality inspection. Box door weld ...,Unclear / insufficient detail,Defective product or part,True,high,No,Description mentions defect evidence such as c...


In [88]:
# ============================================================
# ROOT-CAUSE CLASSIFICATION OUTPUT TABLE
# ============================================================

root_cause_classification_table = df[[
    "incident_id",
    "incident_type_clean",
    "supplier_clean",
    "incident_description",
    "root_cause_category_original_gemini",
    "root_cause_category",
    "root_cause_confidence",
    "root_cause_used_fallback",
    "review_needed",
    "root_cause_justification"
]].copy()

root_cause_classification_table = root_cause_classification_table.rename(columns={
    "incident_id": "Incident ID",
    "incident_type_clean": "Incident Type",
    "supplier_clean": "Supplier",
    "incident_description": "Original Description",
    "root_cause_category_original_gemini": "Original Gemini Category",
    "root_cause_category": "Final Root-Cause Category",
    "root_cause_confidence": "Confidence",
    "root_cause_used_fallback": "Rule-Based Fallback Used",
    "review_needed": "Review Needed",
    "root_cause_justification": "Justification"
})

root_cause_classification_table.head()

,Incident ID,Incident Type,Supplier,Original Description,Original Gemini Category,Final Root-Cause Category,Confidence,Rule-Based Fallback Used,Review Needed,Justification
0,INC_1,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True,No,"Description mentions non-compliance, MIL-STD, ..."
1,INC_2,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True,No,"Description mentions non-compliance, MIL-STD, ..."
2,INC_3,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True,No,"Description mentions non-compliance, MIL-STD, ..."
3,INC_4,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True,No,"Description mentions non-compliance, MIL-STD, ..."
4,INC_5,RMA,Velcronix,Items received were non compliant with MIL-STD...,Unclear / insufficient detail,Poor product quality,high,True,No,"Description mentions non-compliance, MIL-STD, ..."


In [89]:
# ============================================================
# ROOT-CAUSE SUMMARY TABLE
# ============================================================

root_cause_summary = create_count_summary(
    df=df,
    group_col="root_cause_category",
    total_count=len(df)
)

root_cause_summary

,root_cause_category,incident_count,percentage_of_total_incidents
7,Poor product quality,16,32.0
5,Internal warehouse process issue,13,26.0
1,Defective product or part,5,10.0
2,Documentation or labelling issue,5,10.0
6,Material damaged during warehouse handling,4,8.0
8,Unclear / insufficient detail,4,8.0
0,Damaged packaging,1,2.0
4,Incorrect quantity shipped,1,2.0
3,Incorrect item shipped,1,2.0


In [90]:
# ============================================================
# ROOT CAUSE BY SUPPLIER
# ============================================================

root_cause_by_supplier = (
    df.groupby(["root_cause_category", "supplier_clean"], dropna=False)
      .size()
      .reset_index(name="incident_count")
      .sort_values(
          ["root_cause_category", "incident_count"],
          ascending=[True, False]
      )
)

root_cause_by_supplier

,root_cause_category,supplier_clean,incident_count
0,Damaged packaging,CommandOne Defense,1
1,Defective product or part,Aperion Defense,4
2,Defective product or part,StratoSpacer Technologies,1
3,Documentation or labelling issue,Aperion Defense,4
4,Documentation or labelling issue,Viryonox,1
5,Incorrect item shipped,Prixoto Electronics,1
6,Incorrect quantity shipped,Orionist Systems,1
7,Internal warehouse process issue,AeroNovus Instruments,5
9,Internal warehouse process issue,Morogdon Defense,4
10,Internal warehouse process issue,Viryonox,3


In [91]:
# ============================================================
# SUPPLIER VS WAREHOUSE LIKELY DRIVER MAPPING
# Uses both root-cause category and explicit description text
# ============================================================

def assign_likely_driver(row):
    """
    Assigns likely driver based on root-cause category and description wording.
    This should be interpreted as indicative, not definitive.
    """

    category = row["root_cause_category"]
    description = str(row["incident_description"]).lower()

    warehouse_terms = [
        "warehouse team",
        "forklift",
        "dropped from",
        "warehouse shelf",
        "issued from warehouse",
        "warehouse audit",
        "incorrect forklift",
        "new labels were printed",
        "engineering will be rotating stock",
        "screws were issued from warehouse",
        "upon issuing",
        "installed by engineering",
        "previously installed by engineering"
    ]

    supplier_terms = [
        "supplier",
        "vendor",
        "rma",
        "returned and replaced",
        "return and replacement",
        "replacement initiated with supplier",
        "ticket opened with supplier",
        "returned for refund",
        "replacement received",
        "incorrect item received",
        "incorrect qty received"
    ]

    if any(term in description for term in warehouse_terms):
        return "Warehouse-driven"

    if any(term in description for term in supplier_terms):
        return "Supplier-driven"

    if category in [
        "Defective product or part",
        "Incorrect item shipped",
        "Incorrect quantity shipped",
        "Poor product quality",
        "Supplier delivery issue"
    ]:
        return "Supplier-driven"

    if category in [
        "Material damaged during warehouse handling",
        "Internal warehouse process issue"
    ]:
        return "Warehouse-driven"

    if category in [
        "Damaged packaging",
        "Documentation or labelling issue"
    ]:
        return "Mixed"

    return "Unclear"


df["likely_driver"] = df.apply(assign_likely_driver, axis=1)

driver_summary = (
    df.groupby("likely_driver")
      .size()
      .reindex(
          ["Supplier-driven", "Warehouse-driven", "Mixed", "Unclear"],
          fill_value=0
      )
      .reset_index(name="incident_count")
)

driver_summary = add_percentage_column(
    driver_summary,
    count_col="incident_count",
    total_count=len(df)
)

driver_summary

,likely_driver,incident_count,percentage_of_total_incidents
0,Supplier-driven,26,52.0
1,Warehouse-driven,20,40.0
2,Mixed,0,0.0
3,Unclear,4,8.0


In [92]:
# ============================================================
# EXECUTIVE ROOT-CAUSE SUMMARY TABLE
# ============================================================

executive_summary_rows = []

for category in root_cause_summary["root_cause_category"]:
    category_df = df[df["root_cause_category"] == category]

    top_suppliers = (
        category_df["supplier_clean"]
        .value_counts()
        .head(3)
        .index
        .tolist()
    )

    count = len(category_df)
    pct = round(count / len(df) * 100, 2)

    likely_driver = (
        category_df["likely_driver"]
        .value_counts()
        .idxmax()
    )

    executive_summary_rows.append({
        "root_cause_category": category,
        "incident_count": count,
        "percentage_of_total_incidents": pct,
        "top_associated_suppliers": ", ".join(top_suppliers),
        "likely_driver": likely_driver
    })

executive_root_cause_summary = pd.DataFrame(executive_summary_rows)

executive_root_cause_summary

,root_cause_category,incident_count,percentage_of_total_incidents,top_associated_suppliers,likely_driver
0,Poor product quality,16,32.0,"Orionist Systems, Velcronix, Aperion Defense",Supplier-driven
1,Internal warehouse process issue,13,26.0,"AeroNovus Instruments, Morogdon Defense, Viryonox",Warehouse-driven
2,Defective product or part,5,10.0,"Aperion Defense, StratoSpacer Technologies",Supplier-driven
3,Documentation or labelling issue,5,10.0,"Aperion Defense, Viryonox",Supplier-driven
4,Material damaged during warehouse handling,4,8.0,Booftie Technologies,Warehouse-driven
5,Unclear / insufficient detail,4,8.0,Graveson-Reed,Unclear
6,Damaged packaging,1,2.0,CommandOne Defense,Warehouse-driven
7,Incorrect quantity shipped,1,2.0,Orionist Systems,Supplier-driven
8,Incorrect item shipped,1,2.0,Prixoto Electronics,Supplier-driven


In [93]:
# ============================================================
# HUMAN REVIEW TABLE
# ============================================================

human_review_table = df[df["review_needed"] == "Yes"][[
    "incident_id",
    "incident_type_clean",
    "supplier_clean",
    "incident_description",
    "root_cause_category_original_gemini",
    "root_cause_category",
    "root_cause_confidence",
    "root_cause_used_fallback",
    "likely_driver",
    "root_cause_justification"
]].copy()

human_review_table

,incident_id,incident_type_clean,supplier_clean,incident_description,root_cause_category_original_gemini,root_cause_category,root_cause_confidence,root_cause_used_fallback,likely_driver,root_cause_justification
12,INC_13,RMA,Aperion Defense,Order received by warehouse did not have the c...,Unclear / insufficient detail,Documentation or labelling issue,high,True,Supplier-driven,"Description mentions incorrect labels, packing..."
13,INC_14,RMA,Aperion Defense,Order received by warehouse did not have the c...,Unclear / insufficient detail,Documentation or labelling issue,high,True,Supplier-driven,"Description mentions incorrect labels, packing..."
14,INC_15,RMA,Aperion Defense,Order received by warehouse did not have the c...,Unclear / insufficient detail,Documentation or labelling issue,high,True,Supplier-driven,"Description mentions incorrect labels, packing..."
15,INC_16,RMA,Aperion Defense,Order received by warehouse did not have the c...,Unclear / insufficient detail,Documentation or labelling issue,high,True,Supplier-driven,"Description mentions incorrect labels, packing..."
16,INC_17,Work Order,Graveson-Reed,Gear was discoverd to have scuffing issues on ...,Unclear / insufficient detail,Unclear / insufficient detail,low,False,Unclear,No clear root-cause phrase matched.
17,INC_18,Work Order,Graveson-Reed,Gear was discoverd to have scuffing issues on ...,Unclear / insufficient detail,Unclear / insufficient detail,low,False,Unclear,No clear root-cause phrase matched.
18,INC_19,Work Order,Graveson-Reed,Gear was discoverd to have scuffing issues on ...,Unclear / insufficient detail,Unclear / insufficient detail,low,False,Unclear,No clear root-cause phrase matched.
19,INC_20,Work Order,Graveson-Reed,Gear was discoverd to have scuffing issues on ...,Unclear / insufficient detail,Unclear / insufficient detail,low,False,Unclear,No clear root-cause phrase matched.
20,INC_21,Work Order,CommandOne Defense,Item was discovered to have waterstained packa...,Unclear / insufficient detail,Damaged packaging,medium,True,Warehouse-driven,"Description mentions packaging, box damage, or..."
29,INC_30,RMA,Viryonox,Cables were discovered to be incorrectly label...,Unclear / insufficient detail,Documentation or labelling issue,high,True,Warehouse-driven,"Description mentions incorrect labels, packing..."


In [94]:
# ============================================================
# REVIEW SUMMARY
# ============================================================

review_summary = create_count_summary(
    df=df,
    group_col="review_needed",
    total_count=len(df)
)

review_summary

,review_needed,incident_count,percentage_of_total_incidents
0,No,39,78.0
1,Yes,11,22.0


In [95]:
# ============================================================
# EXPORT TEXT ANALYSIS OUTPUTS
# ============================================================

text_analysis_tables = {
    "Root Cause Classifications": root_cause_classification_table,
    "Root Cause Summary": root_cause_summary,
    "Root Cause Supplier": root_cause_by_supplier,
    "Likely Driver Summary": driver_summary,
    "Executive RCA Summary": executive_root_cause_summary,
    "Classification Review": classification_review_table,
    "Human Review Table": human_review_table,
    "Review Summary": review_summary
}

export_tables_to_excel(
    output_file="incident_text_analysis_outputs.xlsx",
    tables=text_analysis_tables
)

Tables exported to: incident_text_analysis_outputs.xlsx


In [96]:
# ============================================================
# FINAL PREVIEW
# ============================================================

df[[
    "incident_id",
    "incident_description",
    "root_cause_category",
    "likely_driver",
    "root_cause_confidence",
    "review_needed",
    "root_cause_justification"
]].head(50)

,incident_id,incident_description,root_cause_category,likely_driver,root_cause_confidence,review_needed,root_cause_justification
0,INC_1,Items received were non compliant with MIL-STD...,Poor product quality,Supplier-driven,high,No,"Description mentions non-compliance, MIL-STD, ..."
1,INC_2,Items received were non compliant with MIL-STD...,Poor product quality,Supplier-driven,high,No,"Description mentions non-compliance, MIL-STD, ..."
2,INC_3,Items received were non compliant with MIL-STD...,Poor product quality,Supplier-driven,high,No,"Description mentions non-compliance, MIL-STD, ..."
3,INC_4,Items received were non compliant with MIL-STD...,Poor product quality,Supplier-driven,high,No,"Description mentions non-compliance, MIL-STD, ..."
4,INC_5,Items received were non compliant with MIL-STD...,Poor product quality,Supplier-driven,high,No,"Description mentions non-compliance, MIL-STD, ..."
5,INC_6,Item received was not compliant with MIL-STD a...,Poor product quality,Supplier-driven,high,No,"Description mentions non-compliance, MIL-STD, ..."
6,INC_7,Item failed quality inspection. Box door weld ...,Defective product or part,Supplier-driven,high,No,Description mentions defect evidence such as c...
7,INC_8,Mounting plates did not have all required scre...,Internal warehouse process issue,Warehouse-driven,high,No,Description mentions a warehouse process issue...
8,INC_9,Item failed quality inspection. Box door weld ...,Defective product or part,Supplier-driven,high,No,Description mentions defect evidence such as c...
9,INC_10,Item failed quality inspection. Box door weld ...,Defective product or part,Supplier-driven,high,No,Description mentions defect evidence such as c...
